In [1]:
from PIL import Image
import xlsxwriter
from io import BytesIO
import pandas as pd
import os
import glob
import pyodbc

Conexão SQL, Query e Tratamento:
Alterar a quantidades de querys necessárias por PCP e a Coleção alvo

In [ ]:
conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=db_name;'
                      'Database=tb_name;'
                      'Trusted_Connection=yes;')

def get_query(COLECAO,CANAL):
    # Usamos f-string para injetar o valor de período_pcp no WHERE
    return f"""
    SELECT *
    FROM TABELA AS T
    WHERE T.COLECAO IN = '{COLECAO}' 
        AND T1.COLECAO LIKE '%{CANAL}%'
        """

COLECAO = 'VER27'  
CANAL = 'VRJ'  #AT, VRJ, FR
query_1 = get_query('PREVIEW',COLECAO,CANAL)

df_1 = pd.read_sql_query(query_1, conn)

Mapeamento de Fotos: Entrada do Caminho de Fotos da Coleção (Necessário alterar apenas o nome da pasta da coleção e eventualmente o ano)

In [ ]:

# 1. Defina o caminho base (o ** e recursive=True buscam em todas as subpastas)
caminho_base = r'\\\**\*.jpg'

mapa_fotos = {os.path.basename(f): f for f in glob.glob(caminho_base, recursive=True)}
print(f"Mapeamento concluído! {len(mapa_fotos)} fotos encontradas.")



Função de Configuração do Excel

In [ ]:
def gerar_catalogo(arquivo,nome_aba,df):
        #Tamanhos da foto
        largura = 304
        altura = 456

        #Coluna do Excel
        largura_coluna = 44


        worksheet = arquivo.add_worksheet(nome_aba) #CRIAR ABA NO EXCEL
        worksheet.hide_gridlines(2)

        # Estilos
        header_format = arquivo.add_format({
            'bold': True, 'align': 'center', 'bg_color': '#F6EAD6', 'font_size': 22
        })
        bold_format = arquivo.add_format({'bold': True, 'align': 'center', 'font_size': 22})
        text_format = arquivo.add_format({'align': 'center','valign':'vcenter', 'font_size': 22, 'text_wrap':True})

        worksheet.set_column('A:J', largura_coluna) # Quantidade de colunas e tamanho da coluna

        col_index = 0  # Index para controlar a coluna inicial e a contagem de colunas
        row_offset = 0 # Controla em qual linha o "bloco" do produto começa

        for row in df.itertuples():
            # --- CABEÇALHO (LINHA) ---
            if row.ROW_LINHA == 1:  #IF utilizando para aparecer no cabeçalho apenas a primeira ocorrência
                worksheet.write(row_offset, col_index, str(row.LINHA), header_format)
            else:
                worksheet.write(row_offset, col_index, '', header_format)
            # --- CABEÇALHO (DESC_COR) ---
            if row.ROW_DESC == 1: #IF utilizando para aparecer no cabeçalho apenas a primeira ocorrência
                worksheet.write(row_offset+1, col_index, str(row.DESC_COR_PRODUTO), header_format)
            else:
                worksheet.write(row_offset+1, col_index, '', header_format)

            # --- IMAGEM ---
            # 1. Definimos o nome do arquivo que buscamos
            nome_esperado = f"{row.PRODUTO}-{row.COR_PRODUTO}.jpg"
            nome_esperado_2 = f"{row.PRODUTO}.{row.COR_PRODUTO}.jpg"

            # No Python, usamos 'None' em vez de 'null'
            nome_final = nome_esperado if nome_esperado in mapa_fotos else nome_esperado_2
            caminho_foto_prod = mapa_fotos.get(nome_final)
            # 2. Consultamos o mapa (Dicionário que você criou antes do loop)
            caminho_foto_prod = mapa_fotos.get(nome_final)

            if caminho_foto_prod:
                try: #Fazemos esse try pois, pode não haver a imagem, aí retornaremos algo ao final
                    with open(caminho_foto_prod, "rb") as f:
                        image_data = BytesIO(f.read())
                        
                    with Image.open(image_data) as img:
                        # Força o redimensionamento físico da foto
                        img_redimensionada = img.resize((largura, altura), Image.LANCZOS)
                        
                        buffer_final = BytesIO()
                        img_redimensionada.save(buffer_final, format="JPEG", quality=95)
                        
                        # ESSENCIAL: Volta para o início do arquivo na memória
                        buffer_final.seek(0)
                
                        # Insere a imagem dentro do bloco para garantir acesso ao buffer
                        worksheet.insert_image(row_offset + 2, col_index, nome_final, {
                            'image_data': buffer_final,
                            'x_scale': 1,
                            'y_scale': 1,
                            'x_offset': 8.8,
                            'y_offset': 5,
                            'object_position': 1,
                            'image_format': 'jpg'
                        })

                    worksheet.set_row(row_offset + 2, 350) #Onde a foto será coloca e altura da célula

                except Exception as e:
                    worksheet.write(row_offset + 2, col_index, "Erro ao carregar", text_format)
                    print(f"Erro na imagem {nome_final}: {e}")
            else:
                # Caso o nome do arquivo não exista no mapa de pastas
                worksheet.write(row_offset + 2, col_index, "Imagem Não Encontrada", text_format)
                worksheet.set_row(row_offset + 2, 350)

            # Escrevemos os dados nas linhas logo abaixo da imagem
            worksheet.write(row_offset + 3, col_index, row.PRODCOR, bold_format)
            worksheet.write(row_offset + 4, col_index, row.DESC_PRODUTO, text_format)
            worksheet.write(row_offset + 5, col_index, row.DESC_COR_PRODUTO, text_format)
            worksheet.write(row_offset + 6, col_index, row.GRADE, text_format)
            worksheet.write(row_offset + 7, col_index, row.VO, text_format)

            # --- LÓGICA DE POSICIONAMENTO (O seu IF contador) ---
            col_index += 1

            if col_index > 9: # Se chegou na 10ª coluna, pula para a próxima linha de blocos
                col_index = 0
                row_offset += 12 # Pula 12 linhas (Espaço entre a primeira linha e a útlima)

        # Configurações de Impressão (Para parecer um PDF)
        worksheet.set_paper(9) # A4
        worksheet.set_landscape() # Ou portrait, conforme preferir
        worksheet.set_margins(left=0.5, right=0.5, top=0.5, bottom=0.5)

Entrada Final: Entrar o nome do arquivo à ser criado e nome de abas por PCPS e Dataframes

In [ ]:
workbook = xlsxwriter.Workbook('BOARD VER27.xlsx')
gerar_catalogo(workbook,'PREVIEW',df_1)
workbook.close()